## Zone 0 Install Dependencies

In [1]:
!apt-get update
!apt-get install -y ffmpeg
!pip install -U openai-whisper jiwer
!pip install -U tqdm
!pip install Montreal-Forced-Aligner --break-system-packages

!mfa model download acoustic french  # Modèle acoustique FR
!mfa model download g2p_en           # Graphème-à-phonème pour anglais (si besoin)

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,328 kB]
Get:13 https://r2u.

### Verify Installation

In [2]:
import whisper
import torch
import subprocess

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Whisper loaded:", whisper.load_model("base") is not None)

# 3. Vérifier l'installation
! mfa --version
mfa models list acoustic
subprocess.run(["ffmpeg", "-version"], capture_output=True)
print("FFmpeg OK")

Torch: 2.9.0+cu126
CUDA available: True


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 152MiB/s]


Whisper loaded: True
FFmpeg OK


### Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Import and Configuration

In [4]:
import json
import subprocess
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass, asdict
import whisper
from jiwer import wer as compute_wer


CONFIG = {
    "cha_dir": Path("/content/drive/MyDrive/asr/data/cha"),
    "audio_dir": Path("/content/drive/MyDrive/asr/data/songs"),
    "output_dir": Path("/content/drive/MyDrive/asr/output/whisper_children_dataset"),
    "whisper_model": "base",
    "sample_size": None,  # None = tous, ou 500 pour tester
    "train_ratio": 0.8,
    "sample_rate": 16000,
    "audio_extensions": [".wav", ".mp3", ".m4a", ".flac"],
    "quality_threshold": 0.9,  # 0.6 (loose) to 0.85 (strict)
    # Whether to compute expensive audio features
    "analyze_audio_features": True,  # Set to False to speed up analysis
    # Analysis output
    "save_quality_visualization": True,
    "save_filtered_segments": True,
}

# CHILDES roles
CHILD_ROLES = {"Target_Child", "Child", "Sibling", "Peer", "Playmate"}
ADULT_ROLES = {"Investigator", "Teacher", "Mother", "Father", "Adult", "Caregiver", "Parent"}



## Zone 1: File Matching (.cha ↔ Audio)


COMPLETE PIPELINE NOTEBOOK
Input: Dataset folders (data/ca + data/songs)
Output: Training dataset for Whisper fine-tuning (children voices only)

Flow:
1. Match .cha ↔ Audio files
2. Extract .cha segments (word-level timestamps)
3. Segment audio files based on timestamps
4. Evaluate Whisper baseline (children only)
5. Calculate WER (children only)
6. Create training dataset (JSONL + metadata)


In [5]:
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Union

@dataclass
class WorSegment:
    speaker: str
    text: str
    words: list  # [(word, start, end)]
    file_name: str = ""  # Ajouter le nom du fichier source


def extract_wor_segments(path: Union[Path, str], debug: bool = False) -> List[WorSegment]:
    """
    Extraire segments %wor d'un fichier .cha

    Args:
        path: Chemin vers un fichier .cha OU un dossier contenant des .cha
        debug: Afficher les infos de parsing

    Returns:
        Liste de WorSegment
    """
    path = Path(path)

    if path.is_dir():
        # Si c'est un dossier, traiter tous les .cha
        return _extract_from_directory(path, debug=debug)
    elif path.is_file():
        # Si c'est un fichier, le traiter
        return _extract_from_file(path, debug=debug)
    else:
        raise FileNotFoundError(f"Chemin invalide: {path}")


def _extract_from_directory(cha_dir: Path, debug: bool = False) -> List[WorSegment]:
    """Extraire de tous les fichiers .cha d'un dossier (récursivement)"""

    # Chercher les .cha dans le dossier ET les sous-dossiers
    cha_files = sorted(cha_dir.glob("*.cha")) + sorted(cha_dir.glob("**/*.cha"))
    # Enlever les doublons
    cha_files = sorted(set(cha_files))

    if not cha_files:
        print(f"Aucun fichier .cha trouvé dans {cha_dir}")
        return []

    if debug:
        print(f"Traitement de {len(cha_files)} fichiers .cha\n")

    all_segments = []

    for cha_file in cha_files:
        if debug:
            print(f" {cha_file.name}...", end=" ")

        segments = _extract_from_file(cha_file, debug=False)
        all_segments.extend(segments)

        if debug:
            print(f"({len(segments)} segments)")

    if debug:
        print(f"\n{'=' * 60}")
        print(f"Total: {len(all_segments)} segments de {len(cha_files)} fichiers")
        print(f"{'=' * 60}\n")

    return all_segments


In [6]:

def _extract_from_file(cha_file: Path, debug: bool = False) -> List[WorSegment]:
    """Extraire de un seul fichier .cha"""

    segments = []
    current_speaker = None
    file_name = cha_file.stem

    with cha_file.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()

            # ── tour principal
            if line.startswith("*"):
                current_speaker = line.split(":", 1)[0].replace("*", "").strip()

            # ── word tier
            elif line.startswith("%wor:"):
                if not current_speaker:
                    continue

                # Extraire la partie après "%wor:"
                wor_content = line.split(":", 1)[1].strip()

                # Nettoyer les caractères de contrôle
                wor_content = wor_content.replace('\x15', '')

                # Parser simple : splitter par espaces et apparier word + timestamp
                tokens = wor_content.split()

                words = []
                i = 0
                while i < len(tokens):
                    token = tokens[i]

                    # Vérifier si c'est un timestamp (format XXXXX_XXXXX)
                    if re.match(r"^\d{5,}_\d{5,}$", token):
                        # C'est un timestamp → l'attacher au mot précédent
                        if words:
                            word, _, _ = words[-1]
                            match = re.match(r"(\d+)_(\d+)", token)
                            if match:
                                start, end = int(match.group(1)), int(match.group(2))
                                words[-1] = (word, start, end)
                        i += 1
                        continue

                    # Sinon, c'est un mot
                    words.append((token, None, None))
                    i += 1

                # Filtrer : garder seulement les mots avec timestamps
                words_with_ts = [(w, s, e) for w, s, e in words if s is not None and e is not None]

                if not words_with_ts:
                    continue

                # Nettoyer le texte : enlever les ponctuations isolées
                clean_words = [w for w, _, _ in words_with_ts if w not in ('?', '.', ',', '!', '+...')]

                if clean_words:
                    clean_text = " ".join(clean_words)

                    segments.append(
                        WorSegment(
                            speaker=current_speaker,
                            text=clean_text,
                            words=words_with_ts,
                            file_name=file_name
                        )
                    )

                    if debug and len(segments) <= 3:
                        print(f"\n {current_speaker}")
                        print(f"   Text: {clean_text[:70]}")
                        print(f"   Words: {words_with_ts[:3]}...")

    if debug:
        print(f"\n{'=' * 60}")
        print(f"Total segments ({file_name}): {len(segments)}")
        print(f"{'=' * 60}")

    return segments

In [7]:
def print_statistics(segments: List[WorSegment]):
    """Afficher statistiques détaillées sur les segments"""

    if not segments:
        print(" Aucun segment trouvé")
        return

    print(f"\n{'=' * 60}")
    print("STATISTIQUES")
    print(f"{'=' * 60}")

    # Par speaker
    by_speaker = {}
    by_file = {}

    for seg in segments:
        # Par speaker
        by_speaker.setdefault(seg.speaker, []).append(seg)

        # Par fichier
        by_file.setdefault(seg.file_name, []).append(seg)

    print(f"\nPar speaker ({len(by_speaker)} speakers):")
    for speaker in sorted(by_speaker.keys()):
        segs = by_speaker[speaker]
        total_duration = sum(s.words[-1][2] - s.words[0][1] for s in segs) / 1000
        print(f"   {speaker:20} {len(segs):3d} segments | {total_duration:6.1f}s audio")

    print(f"\nPar fichier ({len(by_file)} fichiers):")
    for file_name in sorted(by_file.keys()):
        segs = by_file[file_name]
        total_duration = sum(s.words[-1][2] - s.words[0][1] for s in segs) / 1000
        print(f"   {file_name:30} {len(segs):3d} segments | {total_duration:6.1f}s audio")

    # Stats globales
    total_duration = sum(s.words[-1][2] - s.words[0][1] for s in segments) / 1000 / 60
    avg_words = sum(len(s.words) for s in segments) / len(segments)

    print(f"\nGlobales:")
    print(f"   Total segments: {len(segments)}")
    print(f"   Total audio: {total_duration:.1f} minutes")
    print(f"   Mots par segment (moyennes): {avg_words:.1f}")
    print(f"{'=' * 60}\n")


if __name__ == "__main__":
    # Exemple 1: Traiter UN fichier
    print("=" * 60)
    print("EXEMPLE 1: UN FICHIER")
    print("=" * 60)
    segments_single = extract_wor_segments(Path("/content/drive/MyDrive/asr/data/cha/1/01-1a.cha"), debug=True)

    # Exemple 2: Traiter UN DOSSIER ENTIER
    print("\n" + "=" * 60)
    print("EXEMPLE 2: DOSSIER ENTIER")
    print("=" * 60)
    segments_all = extract_wor_segments(Path("/content/drive/MyDrive/asr/data/cha/1/"), debug=True)

    # Afficher statistiques
    print_statistics(segments_all)

    # Exemples
    if segments_all:
        print(f"{'=' * 60}")
        print("EXEMPLES DE SEGMENTS:")
        print(f"{'=' * 60}")
        for i, seg in enumerate(segments_all[:5]):
            print(f"\n[{i + 1}] {seg.speaker} ({seg.file_name})")
            print(f"  Text: {seg.text}")
            if seg.words:
                print(f"  Time: {seg.words[0][1]} → {seg.words[-1][2]} ms")
                print(f"  Words: {seg.words[:3]}...")





EXEMPLE 1: UN FICHIER

 KAT
   Text: un escargot Dylan
   Words: [('un', 10438, 10478), ('escargot', 10739, 10919), ('Dylan', 10919, 11419)]...

 KAT
   Text: comment
   Words: [('comment', 35270, 35770)]...

 WIL
   Text: moi fais la fourmi moi
   Words: [('moi', 38747, 39670), ('fais', 40011, 40612), ('la', 41896, 42136)]...

Total segments (01-1a): 26

EXEMPLE 2: DOSSIER ENTIER
Traitement de 112 fichiers .cha

 01-1a.cha... (26 segments)
 01-1b.cha... (8 segments)
 01-2.cha... (211 segments)
 01-3a.cha... (91 segments)
 01-3b.cha... (71 segments)
 01-3c.cha... (89 segments)
 01-3d.cha... (181 segments)
 01-4.cha... (82 segments)
 02-5.cha... (217 segments)
 02-7.cha... (397 segments)
 02-8.cha... (8 segments)
 02-9a.cha... (224 segments)
 02-9b.cha... (118 segments)
 02-9c.cha... (40 segments)
 02-9d.cha... (138 segments)
 02-9e.cha... (17 segments)
 02-9f.cha... (67 segments)
 03-10a.cha... (75 segments)
 03-10b.cha... (130 segments)
 03-13a.cha... (66 segments)
 03-13b.cha... (98 

In [8]:
def find_matching_files(cha_dir: Path, audio_dir: Path, extensions: List[str]) -> Dict:
    """Match .cha with audio files by relative path (respects subdirectories)"""

    cha_files = sorted(cha_dir.glob("**/*.cha"))
    audio_files = []
    for ext in extensions:
        audio_files.extend(audio_dir.glob(f"**/*{ext}"))

    # Créer dicts: relative_path_with_stem → fichier
    # Exemple: "1/01-1a" pour data/cha/1/01-1a.cha
    cha_by_path = {}
    for f in cha_files:
        relative_stem = str(f.relative_to(cha_dir).with_suffix(""))  # "1/01-1a"
        cha_by_path[relative_stem] = f

    audio_by_path = {}
    for f in audio_files:
        relative_stem = str(f.relative_to(audio_dir).with_suffix(""))
        audio_by_path[relative_stem] = f

    # Matcher: chercher les mêmes chemins relatifs
    matched = []
    cha_missing = []
    audio_orphans = []

    for relative_path in cha_by_path:
        if relative_path in audio_by_path:
            matched.append((cha_by_path[relative_path], audio_by_path[relative_path]))
        else:
            cha_missing.append(cha_by_path[relative_path])

    for relative_path in audio_by_path:
        if relative_path not in cha_by_path:
            audio_orphans.append(audio_by_path[relative_path])

    return {
        "matched": matched,
        "cha_missing_audio": cha_missing,
        "audio_orphans": audio_orphans,
        "total_cha": len(cha_files),
        "total_audio": len(audio_files),
        "matched_count": len(matched)
    }


def print_matching_report(result: Dict):
    """Afficher le rapport de matching"""
    print("\n" + "="*70)
    print("STEP 1: FILE MATCHING (by relative path)")
    print("="*70)
    print(f"\n Found:")
    print(f"   Total .cha files:       {result['total_cha']}")
    print(f"   Total audio files:      {result['total_audio']}")
    print(f"   Matched pairs:        {result['matched_count']}")
    print(f"   .cha missing audio:  {len(result['cha_missing_audio'])}")
    print(f"   Audio orphans:       {len(result['audio_orphans'])}")

    if result['cha_missing_audio']:
        print(f"\n   Missing audio for:")
        for cha in result['cha_missing_audio'][:10]:
            print(f"      - {cha.name}")
        if len(result['cha_missing_audio']) > 10:
            print(f"      ... and {len(result['cha_missing_audio']) - 10} more")

    if result['audio_orphans']:
        print(f"\n   Audio without .cha:")
        for audio in result['audio_orphans'][:10]:
            print(f"      - {audio.name}")
        if len(result['audio_orphans']) > 10:
            print(f"      ... and {len(result['audio_orphans']) - 10} more")

    print("\n" + "="*70 + "\n")


In [9]:
  # ZONE 1: Matching
print("\n" + "="*70)
print("ZONE 1: FILE MATCHING")
print("="*70)
match_result = find_matching_files(CONFIG["cha_dir"], CONFIG["audio_dir"], CONFIG["audio_extensions"])
print_matching_report(match_result)

if not match_result["matched"]:
    print("❌ No matched pairs found!")



ZONE 1: FILE MATCHING

STEP 1: FILE MATCHING (by relative path)

 Found:
   Total .cha files:       247
   Total audio files:      245
   Matched pairs:        245
   .cha missing audio:  2
   Audio orphans:       0

   Missing audio for:
      - 03-13c.cha
      - 05-23a.cha



## Zone 2: Segment Extraction (word-level)

In [10]:
def extract_segments_from_matched(matched_pairs: List[Tuple[Path, Path]]) -> List[WorSegment]:
    """Extract .cha segments from matched files only"""

    print("="*70)
    print("STEP 2: EXTRACT .CHA SEGMENTS")
    print("="*70)

    all_segments = []

    for i, (cha_file, audio_file) in enumerate(matched_pairs):
        segments = extract_wor_segments(cha_file, debug=False)
        all_segments.extend(segments)

        if (i + 1) % 50 == 0:
            print(f"  ✓ {i + 1}/{len(matched_pairs)} files")

    print(f"\nExtracted {len(all_segments)} segments with timestamps\n")
    return all_segments

In [11]:
 # ZONE 2: Extract segments from .cha files
print("\n" + "="*70)
print("ZONE 2: EXTRACT SEGMENTS FROM .CHA")
print("="*70)
segments = extract_segments_from_matched(match_result["matched"])




ZONE 2: EXTRACT SEGMENTS FROM .CHA
STEP 2: EXTRACT .CHA SEGMENTS
  ✓ 50/245 files
  ✓ 100/245 files
  ✓ 150/245 files
  ✓ 200/245 files

Extracted 29272 segments with timestamps


## Zone 3: Montreal Forced Aligner (MFA)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# ZONE 3: MFA ALIGNMENT (MISSING IN CURRENT PIPELINE - MUST ADD)
# ════════════════════════════════════════════════════════════════════════

from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from pathlib import Path
import json
import subprocess
from datetime import datetime

@dataclass
class MFASegment:
    """Segment with acoustic alignment from Montreal Forced Aligner"""
    text: str
    speaker: str
    original_start_ms: float  # From .cha
    original_end_ms: float    # From .cha
    mfa_start_ms: float       # From MFA (acoustic)
    mfa_end_ms: float         # From MFA (acoustic)
    confidence: float         # MFA confidence score
    word_times: List[Dict]    # [{'word': str, 'start': ms, 'end': ms, 'confidence': float}, ...]
    audio_file: Path
    textgrid_file: Optional[Path] = None
    
    @property
    def mfa_duration_ms(self) -> float:
        return self.mfa_end_ms - self.mfa_start_ms


class MFAAligner:
    """Wrapper for Montreal Forced Aligner"""
    
    def __init__(self, mfa_model_path: Path = None, tmp_dir: Path = None):
        self.mfa_model = mfa_model_path or Path("pretrained_models/english_us_arpa")
        self.tmp_dir = tmp_dir or Path("./mfa_temp")
        self.tmp_dir.mkdir(exist_ok=True)
        
    def prepare_for_mfa(self, 
                       word_segments: List, 
                       audio_files: List[Path]) -> Path:
        """
        Prepare TextGrid files for MFA from word segments.
        
        Montreal Forced Aligner expects:
        - Audio files in one directory
        - TextGrid/label files with same name in another directory
        
        Args:
            word_segments: List of segments with text (from ZONE 2)
            audio_files: List of audio file paths
            
        Returns:
            Path to directory with prepared TextGrid files
        """
        
        textgrid_dir = self.tmp_dir / "textgrids"
        textgrid_dir.mkdir(exist_ok=True)
        
        # Group segments by audio file
        segments_by_audio = {}
        for segment in word_segments:
            audio_path = segment.audio_file
            if audio_path not in segments_by_audio:
                segments_by_audio[audio_path] = []
            segments_by_audio[audio_path].append(segment)
        
        # Create TextGrid/lab files for each audio
        for audio_path, segments in segments_by_audio.items():
            # MFA-compatible format: one label per line
            # Format: <start_ms> <end_ms> <text>
            lab_content = []
            for seg in segments:
                # Convert ms to seconds for TextGrid format
                start_sec = seg.original_start_ms / 1000.0
                end_sec = seg.original_end_ms / 1000.0
                lab_content.append(f"{start_sec:.3f}\t{end_sec:.3f}\t{seg.text}")
            
            # Save as .lab file (MFA-compatible)
            lab_file = textgrid_dir / f"{audio_path.stem}.lab"
            with open(lab_file, 'w', encoding='utf-8') as f:
                f.write('\n'.join(lab_content))
        
        return textgrid_dir
    
    def run_mfa_alignment(self, 
                          audio_dir: Path, 
                          textgrid_dir: Path,
                          output_dir: Path) -> bool:
        """
        Execute Montreal Forced Aligner.
        
        Args:
            audio_dir: Directory with .wav files
            textgrid_dir: Directory with prepared TextGrid files
            output_dir: Where to save MFA output
            
        Returns:
            True if alignment succeeded
        """
        
        output_dir.mkdir(exist_ok=True)
        
        # MFA command
        cmd = [
            'mfa', 'align',
            str(audio_dir),
            str(textgrid_dir),
            str(self.mfa_model),
            str(output_dir),
            '--clean',
            '--verbose'
        ]
        
        try:
            print(f"Running MFA: {' '.join(cmd)}")
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
            print(f"✅ MFA alignment successful")
            return True
        except subprocess.CalledProcessError as e:
            print(f"❌ MFA alignment failed:")
            print(e.stderr)
            return False
    
    def extract_mfa_timestamps(self, 
                               word_segments: List,
                               textgrid_dir: Path) -> List[MFASegment]:
        """
        Extract word-level timestamps from MFA TextGrid output.
        
        Args:
            word_segments: Original segments from ZONE 2
            textgrid_dir: Directory with MFA output TextGrids
            
        Returns:
            List of MFASegment with acoustic timestamps
        """
        
        mfa_segments = []
        
        for segment in word_segments:
            audio_stem = segment.audio_file.stem
            textgrid_path = textgrid_dir / f"{audio_stem}.TextGrid"
            
            if not textgrid_path.exists():
                print(f"⚠️  No TextGrid for {audio_stem}, skipping")
                continue
            
            # Parse TextGrid (simplified - use textgrid library in production)
            # TextGrid format:
            # File type = "ooTextFile"
            # Object class = "TextGrid"
            # ...
            # xmin = <start>
            # xmax = <end>
            # ...
            # intervals []:
            #   size = N
            #   intervals [1]:
            #     xmin = <word_start>
            #     xmax = <word_end>
            #     text = "<word>"
            
            try:
                word_times = self._parse_textgrid(textgrid_path, segment.text)
                
                # Calculate MFA timestamps
                if word_times:
                    mfa_start = word_times[0]['start_ms']
                    mfa_end = word_times[-1]['end_ms']
                    confidence = sum(w.get('confidence', 1.0) 
                                   for w in word_times) / len(word_times)
                else:
                    mfa_start = segment.original_start_ms
                    mfa_end = segment.original_end_ms
                    confidence = 0.0
                
                # Create MFA segment
                mfa_segment = MFASegment(
                    text=segment.text,
                    speaker=segment.speaker,
                    original_start_ms=segment.original_start_ms,
                    original_end_ms=segment.original_end_ms,
                    mfa_start_ms=mfa_start,
                    mfa_end_ms=mfa_end,
                    confidence=confidence,
                    word_times=word_times,
                    audio_file=segment.audio_file,
                    textgrid_file=textgrid_path
                )
                mfa_segments.append(mfa_segment)
                
            except Exception as e:
                print(f"⚠️  Error parsing TextGrid for {audio_stem}: {e}")
                continue
        
        return mfa_segments
    
    def _parse_textgrid(self, textgrid_path: Path, expected_text: str) -> List[Dict]:
        """
        Parse TextGrid file and extract word-level timestamps.
        
        In production, use: textgrid.TextGrid.load(textgrid_path)
        
        Returns:
            List of {'word': str, 'start_ms': float, 'end_ms': float, 'confidence': float}
        """
        # Simplified placeholder - use textgrid library
        # This should parse MFA's TextGrid output
        return []
    
    def align_segments(self, 
                      word_segments: List,
                      audio_dir: Path,
                      audio_files: List[Path],
                      cache_dir: Optional[Path] = None) -> List[MFASegment]:
        """
        Complete MFA alignment pipeline.
        
        Args:
            word_segments: Segments from ZONE 2 Extract
            audio_dir: Directory containing audio files
            audio_files: List of audio file paths
            cache_dir: Optional directory to cache MFA output
            
        Returns:
            List of MFA-aligned segments with word-level timestamps
        """
        
        print("\n" + "="*80)
        print("ZONE 3: MFA ALIGNMENT")
        print("="*80)
        
        # Step 1: Prepare TextGrids
        print("\n1️⃣  Preparing TextGrid files for MFA...")
        textgrid_dir = self.prepare_for_mfa(word_segments, audio_files)
        print(f"   ✅ Created {len(list(textgrid_dir.glob('*.lab')))} lab files")
        
        # Step 2: Run MFA
        print("\n2️⃣  Running Montreal Forced Aligner...")
        output_dir = cache_dir or self.tmp_dir / "mfa_output"
        success = self.run_mfa_alignment(audio_dir, textgrid_dir, output_dir)
        
        if not success:
            print("   ❌ MFA alignment failed, falling back to .cha timestamps")
            # Fallback: return original segments as MFA segments
            return [MFASegment(
                text=s.text,
                speaker=s.speaker,
                original_start_ms=s.original_start_ms,
                original_end_ms=s.original_end_ms,
                mfa_start_ms=s.original_start_ms,
                mfa_end_ms=s.original_end_ms,
                confidence=0.5,
                word_times=[],
                audio_file=s.audio_file
            ) for s in word_segments]
        
        # Step 3: Extract timestamps
        print("\n3️⃣  Extracting MFA timestamps...")
        mfa_segments = self.extract_mfa_timestamps(word_segments, output_dir)
        print(f"   ✅ Extracted {len(mfa_segments)} MFA-aligned segments")
        
        # Step 4: Statistics
        print("\n4️⃣  MFA Alignment Statistics:")
        if mfa_segments:
            avg_confidence = sum(s.confidence for s in mfa_segments) / len(mfa_segments)
            avg_duration = sum(s.mfa_duration_ms for s in mfa_segments) / len(mfa_segments)
            print(f"   • Average confidence: {avg_confidence:.3f}")
            print(f"   • Average segment duration: {avg_duration:.0f} ms")
            
            # Check for large shifts
            large_shifts = [s for s in mfa_segments 
                           if abs(s.mfa_start_ms - s.original_start_ms) > 500]
            if large_shifts:
                print(f"   ⚠️  {len(large_shifts)} segments with >500ms shift")
        
        return mfa_segments


# ════════════════════════════════════════════════════════════════════════
# ZONE 3: MAIN PIPELINE INTEGRATION
# ════════════════════════════════════════════════════════════════════════

# Dans le main pipeline (après ZONE 2):
if __name__ == "__main__":
    
    # ... ZONE 1 & 2 complétées ...
    
    # ZONE 2: Extract segments from .cha
    word_segments = extract_segments_from_matched(matched_pairs)
    print(f"Extracted {len(word_segments)} segments from .cha files")
    
    # 🆕 ZONE 3: MFA ALIGNMENT (AJOUTER)
    mfa_aligner = MFAAligner(mfa_model_path=Path("pretrained_models/english_us_arpa"))
    mfa_segments = mfa_aligner.align_segments(
        word_segments=word_segments,
        audio_dir=audio_data_dir,
        audio_files=list(audio_data_dir.glob("*.wav"))
    )
    # Output: List[MFASegment] avec:
    # - mfa_start_ms, mfa_end_ms (timestamps acoustiques fiables)
    # - word_times (alignement mot-à-mot from MFA)
    # - confidence score
    
    # ZONE 4: Audio Segmentation (utilise mfa_segments, pas word_segments)
    audio_segments = segment_audio_files_with_mfa(
        mfa_segments=mfa_segments,  # ← UTILISE MFA, pas raw .cha
        audio_dir=audio_data_dir,
        output_dir=segmented_audio_dir
    )

## ZONE 4: Audio Segmentation (using MFA timestamps)

In [12]:
"""
ZONE 4: AUDIO SEGMENTATION (AVEC MFA TIMESTAMPS)

Input: MFASegment[] from Zone 3 (MFA Alignment)
Process: Cut audio files at MFA-aligned boundaries
Output: AudioSegment[] with aligned audio chunks and metadata

Key difference from old version:
- Uses MFA timestamps (acoustic alignment) instead of raw .cha timestamps
- Preserves word-level timing information
- Maintains alignment metadata for downstream processing
"""

from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm
import logging

# ════════════════════════════════════════════════════════════════════════════
# DATA CLASSES
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class AudioSegment:
    """Audio chunk with alignment information"""
    
    # Original segment info
    original_segment_id: str  # Unique identifier
    text: str                 # Transcription text
    speaker: str              # Speaker ID
    
    # MFA-aligned timestamps (from Zone 3)
    mfa_start_ms: float       # Start time in audio (milliseconds)
    mfa_end_ms: float         # End time in audio (milliseconds)
    mfa_duration_ms: float    # Duration of audio segment
    
    # Word-level alignment (from Zone 3 MFA)
    word_times: List[Dict]    # [{'word': str, 'start_ms': float, 'end_ms': float, 'confidence': float}, ...]
    
    # Audio file information
    audio_file: Path          # Path to original audio file
    audio_data: np.ndarray    # Audio waveform (mono)
    sample_rate: int          # Sample rate (Hz)
    
    # Output file information
    output_file: Optional[Path] = None  # Path to saved segment
    
    # Metadata
    confidence: float = 0.0   # MFA confidence score
    num_words: int = 0        # Number of words in segment
    
    def __post_init__(self):
        """Calculate derived values"""
        self.mfa_duration_ms = self.mfa_end_ms - self.mfa_start_ms
        self.num_words = len(self.word_times)
        
        # Calculate average confidence from word-level scores
        if self.word_times:
            self.confidence = sum(w.get('confidence', 1.0) 
                                 for w in self.word_times) / len(self.word_times)
        else:
            self.confidence = 0.5
    
    @property
    def duration_sec(self) -> float:
        """Duration in seconds"""
        return self.mfa_duration_ms / 1000.0
    
    @property
    def num_samples(self) -> int:
        """Number of audio samples"""
        return len(self.audio_data)
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for JSON serialization"""
        return {
            'original_segment_id': self.original_segment_id,
            'text': self.text,
            'speaker': self.speaker,
            'mfa_start_ms': float(self.mfa_start_ms),
            'mfa_end_ms': float(self.mfa_end_ms),
            'mfa_duration_ms': float(self.mfa_duration_ms),
            'duration_sec': self.duration_sec,
            'word_count': self.num_words,
            'confidence': float(self.confidence),
            'audio_file': str(self.audio_file),
            'output_file': str(self.output_file) if self.output_file else None,
            'sample_rate': self.sample_rate,
            'num_samples': self.num_samples,
            'word_times': self.word_times
        }


# ════════════════════════════════════════════════════════════════════════════
# SEGMENTATION FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def load_audio_file(audio_path: Path, sr: Optional[int] = None) -> Tuple[np.ndarray, int]:
    """
    Load audio file using librosa.
    
    Args:
        audio_path: Path to audio file (.wav, .mp3, etc.)
        sr: Target sample rate (None = native rate)
        
    Returns:
        (audio_data, sample_rate) tuple
        - audio_data: mono waveform as numpy array
        - sample_rate: sample rate in Hz
    """
    try:
        y, sr = librosa.load(str(audio_path), sr=sr, mono=True)
        return y, sr
    except Exception as e:
        logging.error(f"Failed to load audio {audio_path}: {e}")
        raise


def ms_to_samples(time_ms: float, sample_rate: int) -> int:
    """Convert milliseconds to audio sample index"""
    return int((time_ms / 1000.0) * sample_rate)


def samples_to_ms(samples: int, sample_rate: int) -> float:
    """Convert audio sample index to milliseconds"""
    return (samples / sample_rate) * 1000.0


def cut_audio_segment(
    audio_data: np.ndarray,
    sample_rate: int,
    start_ms: float,
    end_ms: float,
    pad_ms: float = 0
) -> np.ndarray:
    """
    Cut audio segment from waveform.
    
    Args:
        audio_data: Mono waveform
        sample_rate: Sample rate in Hz
        start_ms: Start time in milliseconds
        end_ms: End time in milliseconds
        pad_ms: Optional padding before start (milliseconds)
        
    Returns:
        Audio segment as numpy array
    """
    # Convert to samples with padding
    start_samples = max(0, ms_to_samples(start_ms - pad_ms, sample_rate))
    end_samples = min(len(audio_data), ms_to_samples(end_ms, sample_rate))
    
    return audio_data[start_samples:end_samples]


def save_audio_segment(
    audio_data: np.ndarray,
    sample_rate: int,
    output_path: Path,
    overwrite: bool = False
) -> Path:
    """
    Save audio segment to file.
    
    Args:
        audio_data: Audio waveform
        sample_rate: Sample rate
        output_path: Output file path (.wav)
        overwrite: Overwrite existing file
        
    Returns:
        Path to saved file
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    if output_path.exists() and not overwrite:
        logging.warning(f"File exists, skipping: {output_path}")
        return output_path
    
    try:
        sf.write(str(output_path), audio_data, sample_rate)
        return output_path
    except Exception as e:
        logging.error(f"Failed to save audio {output_path}: {e}")
        raise


def segment_audio_files_with_mfa(
    mfa_segments: List,
    audio_dir: Path,
    output_dir: Path,
    pad_ms: float = 50,
    overwrite: bool = False,
    target_sr: Optional[int] = None
) -> List[AudioSegment]:
    """
    Segment audio files using MFA-aligned timestamps.
    
    🔑 KEY: Uses MFA timestamps (acoustic alignment) instead of raw .cha times
    
    Args:
        mfa_segments: List of MFASegment from Zone 3
                     (must contain mfa_start_ms, mfa_end_ms, word_times)
        audio_dir: Directory containing audio files
        output_dir: Directory to save segmented audio
        pad_ms: Padding before segment start (for context)
        overwrite: Overwrite existing segment files
        target_sr: Resample audio to this rate (None = keep original)
        
    Returns:
        List of AudioSegment with audio data and metadata
    """
    
    print("\n" + "="*80)
    print("ZONE 4: AUDIO SEGMENTATION (WITH MFA TIMESTAMPS)")
    print("="*80)
    
    output_dir.mkdir(parents=True, exist_ok=True)
    audio_segments = []
    
    # Group segments by audio file for efficient loading
    segments_by_audio = {}
    for seg in mfa_segments:
        audio_path = seg.audio_file
        if audio_path not in segments_by_audio:
            segments_by_audio[audio_path] = []
        segments_by_audio[audio_path].append(seg)
    
    # Process each audio file
    print(f"\n1️⃣  Processing {len(segments_by_audio)} audio files...")
    
    for audio_path, segments in tqdm(segments_by_audio.items(), desc="Audio files"):
        
        # Load audio once per file
        try:
            audio_data, sr = load_audio_file(audio_path, sr=target_sr)
        except Exception as e:
            logging.error(f"Skipping file {audio_path}: {e}")
            continue
        
        # Segment each MFA segment
        for seg_idx, mfa_seg in enumerate(segments):
            try:
                # Create unique output filename
                segment_name = f"{audio_path.stem}_seg{seg_idx:04d}.wav"
                segment_output = output_dir / segment_name
                
                # Cut audio using MFA timestamps
                audio_chunk = cut_audio_segment(
                    audio_data=audio_data,
                    sample_rate=sr,
                    start_ms=mfa_seg.mfa_start_ms,
                    end_ms=mfa_seg.mfa_end_ms,
                    pad_ms=pad_ms
                )
                
                # Verify audio chunk length
                duration_sec = len(audio_chunk) / sr
                if duration_sec < 0.1:  # Skip very short segments
                    logging.warning(f"Skipping very short segment: {duration_sec:.2f}s")
                    continue
                
                # Save audio chunk
                save_audio_segment(
                    audio_data=audio_chunk,
                    sample_rate=sr,
                    output_path=segment_output,
                    overwrite=overwrite
                )
                
                # Create AudioSegment object
                audio_segment = AudioSegment(
                    original_segment_id=f"{audio_path.stem}_seg{seg_idx:04d}",
                    text=mfa_seg.text,
                    speaker=mfa_seg.speaker,
                    mfa_start_ms=mfa_seg.mfa_start_ms,
                    mfa_end_ms=mfa_seg.mfa_end_ms,
                    word_times=mfa_seg.word_times.copy(),  # PRESERVE word alignment from MFA
                    audio_file=audio_path,
                    audio_data=audio_chunk,
                    sample_rate=sr,
                    output_file=segment_output,
                    confidence=mfa_seg.confidence
                )
                
                audio_segments.append(audio_segment)
                
            except Exception as e:
                logging.error(f"Error segmenting {audio_path} segment {seg_idx}: {e}")
                continue
    
    # Statistics
    print(f"\n2️⃣  Segmentation Statistics:")
    print(f"   ✅ Created {len(audio_segments)} audio segments")
    
    if audio_segments:
        durations = [seg.duration_sec for seg in audio_segments]
        print(f"   • Average duration: {np.mean(durations):.2f}s")
        print(f"   • Min duration: {np.min(durations):.2f}s")
        print(f"   • Max duration: {np.max(durations):.2f}s")
        
        avg_confidence = np.mean([seg.confidence for seg in audio_segments])
        print(f"   • Average MFA confidence: {avg_confidence:.3f}")
    
    return audio_segments


# ════════════════════════════════════════════════════════════════════════════
# VALIDATION FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def validate_audio_segment(audio_seg: AudioSegment) -> bool:
    """
    Validate audio segment for correctness.
    
    Checks:
    1. Audio data is valid (not empty)
    2. Audio duration matches timestamp duration
    3. Word times are within segment boundaries
    4. Word sequence is monotonic
    5. Number of words matches text
    """
    
    reasons = []
    
    # Check 1: Audio data exists and is valid
    if audio_seg.audio_data is None or len(audio_seg.audio_data) == 0:
        reasons.append("empty_audio")
    
    # Check 2: Audio duration matches timestamps
    audio_duration_ms = (len(audio_seg.audio_data) / audio_seg.sample_rate) * 1000
    expected_duration_ms = audio_seg.mfa_duration_ms
    
    if abs(audio_duration_ms - expected_duration_ms) > 100:  # 100ms tolerance
        reasons.append(f"duration_mismatch({audio_duration_ms:.0f}ms vs {expected_duration_ms:.0f}ms)")
    
    # Check 3: Word times within boundaries
    if audio_seg.word_times:
        word_starts = [w['start_ms'] for w in audio_seg.word_times]
        word_ends = [w['end_ms'] for w in audio_seg.word_times]
        
        # All word times should be within segment times
        if min(word_starts) < audio_seg.mfa_start_ms:
            reasons.append("word_time_before_start")
        
        if max(word_ends) > audio_seg.mfa_end_ms:
            reasons.append("word_time_after_end")
        
        # Check 4: Monotonic progression
        for i in range(len(audio_seg.word_times) - 1):
            w1_end = audio_seg.word_times[i]['end_ms']
            w2_start = audio_seg.word_times[i+1]['start_ms']
            
            if w1_end > w2_start:
                reasons.append(f"word_times_overlap({i},{i+1})")
                break
    
    # Check 5: Word count consistency
    word_count_text = len(audio_seg.text.split())
    word_count_times = len(audio_seg.word_times)
    
    if abs(word_count_text - word_count_times) > 1:  # 1-word tolerance
        reasons.append(f"word_count_mismatch({word_count_text} vs {word_count_times})")
    
    return len(reasons) == 0


def validate_segmentation_output(audio_segments: List[AudioSegment]) -> Dict:
    """
    Validate entire segmentation output.
    
    Returns:
        Dictionary with validation results
    """
    
    print("\n3️⃣  Validating segmentation...")
    
    valid_count = 0
    invalid_segments = []
    
    for seg in audio_segments:
        if validate_audio_segment(seg):
            valid_count += 1
        else:
            invalid_segments.append(seg.original_segment_id)
    
    results = {
        'total': len(audio_segments),
        'valid': valid_count,
        'invalid': len(invalid_segments),
        'invalid_list': invalid_segments,
        'success_rate': (valid_count / len(audio_segments) * 100) if audio_segments else 0
    }
    
    print(f"   ✅ Valid: {valid_count}/{len(audio_segments)} ({results['success_rate']:.1f}%)")
    
    if invalid_segments:
        print(f"   ⚠️  Invalid segments: {len(invalid_segments)}")
        for seg_id in invalid_segments[:5]:  # Show first 5
            print(f"      └─ {seg_id}")
    
    return results


# ════════════════════════════════════════════════════════════════════════════
# OUTPUT FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════

def save_segmentation_manifest(
    audio_segments: List[AudioSegment],
    output_file: Path
) -> Path:
    """
    Save segmentation manifest to JSON file.
    
    Contains metadata for all segments for easy reference.
    
    Args:
        audio_segments: List of AudioSegment
        output_file: Path to output JSON file
        
    Returns:
        Path to saved manifest
    """
    
    manifest = {
        'version': '1.0',
        'segment_count': len(audio_segments),
        'segments': [seg.to_dict() for seg in audio_segments]
    }
    
    import json
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    
    return output_file


def print_segmentation_summary(audio_segments: List[AudioSegment]):
    """Print summary of segmentation results"""
    
    if not audio_segments:
        print("⚠️  No segments created")
        return
    
    print("\n4️⃣  Segmentation Summary:")
    print(f"   • Total segments: {len(audio_segments)}")
    
    # Duration statistics
    durations = [seg.duration_sec for seg in audio_segments]
    print(f"   • Duration range: {np.min(durations):.2f}s - {np.max(durations):.2f}s")
    print(f"   • Average duration: {np.mean(durations):.2f}s")
    print(f"   • Total audio: {np.sum(durations):.1f}s")
    
    # Word statistics
    word_counts = [seg.num_words for seg in audio_segments]
    print(f"   • Words per segment: {np.mean(word_counts):.1f} avg")
    
    # Speaker distribution
    speakers = {}
    for seg in audio_segments:
        speakers[seg.speaker] = speakers.get(seg.speaker, 0) + 1
    
    print(f"   • Speakers: {len(speakers)}")
    for speaker, count in sorted(speakers.items(), key=lambda x: x[1], reverse=True):
        print(f"      └─ {speaker}: {count} segments")
    
    # MFA confidence
    confidences = [seg.confidence for seg in audio_segments]
    print(f"   • MFA confidence: {np.mean(confidences):.3f} avg")


# ════════════════════════════════════════════════════════════════════════════
# MAIN ZONE 4 FUNCTION
# ════════════════════════════════════════════════════════════════════════════

def run_zone_4_audio_segmentation(
    mfa_segments: List,
    audio_dir: Path,
    output_dir: Path,
    target_sr: Optional[int] = 16000,
    pad_ms: float = 50
) -> Tuple[List[AudioSegment], Dict]:
    """
    Complete Zone 4: Audio Segmentation pipeline.
    
    🔑 KEY DIFFERENCES FROM OLD VERSION:
    - Input: MFASegment[] (with acoustic timestamps)
    - NOT: WordSegment[] (with raw .cha timestamps)
    - Output: AudioSegment[] with audio data + MFA alignment preserved
    
    Args:
        mfa_segments: List of MFASegment from Zone 3 (WITH MFA timestamps)
        audio_dir: Directory containing original audio files
        output_dir: Directory to save segmented audio
        target_sr: Resample to this sample rate (16kHz recommended)
        pad_ms: Padding before segment (for context)
        
    Returns:
        (audio_segments, validation_results) tuple
    """
    
    print("\n" + "="*80)
    print("ZONE 4: AUDIO SEGMENTATION WITH MFA TIMESTAMPS")
    print("="*80)
    
    # Segment audio using MFA timestamps
    audio_segments = segment_audio_files_with_mfa(
        mfa_segments=mfa_segments,
        audio_dir=audio_dir,
        output_dir=output_dir,
        pad_ms=pad_ms,
        target_sr=target_sr
    )
    
    # Validate output
    validation = validate_segmentation_output(audio_segments)
    
    # Print summary
    print_segmentation_summary(audio_segments)
    
    # Save manifest
    manifest_path = output_dir / "segmentation_manifest.json"
    save_segmentation_manifest(audio_segments, manifest_path)
    print(f"\n   ✅ Manifest saved: {manifest_path}")
    
    return audio_segments, validation


# ════════════════════════════════════════════════════════════════════════════
# EXAMPLE USAGE
# ════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    
    # Setup logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    
    # Example paths (adjust to your setup)
    audio_data_dir = Path("data/audio")
    segmented_output_dir = Path("output/segmented_audio")
    
    # This would be called after Zone 3 (MFA Alignment)
    # mfa_segments would come from: mfa_aligner.align_segments(...)
    
    # Example call:
    # audio_segments, validation = run_zone_4_audio_segmentation(
    #     mfa_segments=mfa_segments,  # From Zone 3
    #     audio_dir=audio_data_dir,
    #     output_dir=segmented_output_dir,
    #     target_sr=16000  # Resample to 16kHz
    # )
    
    print("Zone 4 module loaded successfully")
    print("Ready to segment audio with MFA timestamps")

## ZONE 5: Extract Speaker Metadata 

In [14]:
def extract_all_speakers_info(matched_pairs: List[Tuple[Path, Path]]) -> Dict[str, str]:
    """Extract speaker roles from all .cha files"""

    all_speakers = {}

    for cha_file, _ in matched_pairs:
        with cha_file.open(encoding="utf-8") as f:
            for line in f:
                if line.startswith("@Participants:"):
                    participants_str = line.split(":", 1)[1].strip()
                    for participant in participants_str.split(","):
                        participant = participant.strip()
                        parts = participant.rsplit(" ", 1)
                        if len(parts) == 2:
                            speaker_name, role = parts
                            all_speakers[speaker_name] = role

    return all_speakers

In [15]:
  # ZONE 4: Get speaker info
print("\n" + "="*70)
print("ZONE 4: EXTRACT SPEAKER ROLES")
print("="*70)
speakers_info = extract_all_speakers_info(match_result["matched"])


ZONE 4: EXTRACT SPEAKER ROLES


## Zone 6 : Merge segment

In [ ]:

from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
import numpy as np


@dataclass
class MergedSegment:
    """Multiple segments merged to target duration, with preserved alignment"""
    texts: List[str]  # Original texts of merged segments
    merged_text: str  # Concatenated text
    speaker: str
    original_segments: List  # Reference to originals (AudioSegment)
    
    mfa_start_ms: float  # Start of first segment
    mfa_end_ms: float    # End of last segment
    merged_duration_ms: float  # Total duration
    
    # CRITICAL: Preserve exact word-level alignment after merge
    word_times_merged: List[Dict]  # [{'word': str, 'start': ms, 'end': ms, ...}, ...]
    
    audio_file: Path
    
    @property
    def duration_sec(self) -> float:
        """Duration in seconds"""
        return self.merged_duration_ms / 1000.0
    
    @property
    def num_words(self) -> int:
        """Number of words"""
        return len(self.word_times_merged)
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for JSON"""
        return {
            'merged_text': self.merged_text,
            'speaker': self.speaker,
            'mfa_start_ms': float(self.mfa_start_ms),
            'mfa_end_ms': float(self.mfa_end_ms),
            'duration_sec': self.duration_sec,
            'num_words': self.num_words,
            'num_segments': len(self.original_segments),
            'word_times': self.word_times_merged,
            'audio_file': str(self.audio_file)
        }


def _merge_segment_group(segment_group: List) -> MergedSegment:
    """
    Merge a group of consecutive audio segments.
    ⚠️  PRESERVE exact word-level alignment.
    """
    
    # Concatenate texts
    texts = [s.text for s in segment_group]
    merged_text = " ".join(texts)
    
    # Merge word times: collect all word timestamps in order
    merged_word_times = []
    for segment in segment_group:
        merged_word_times.extend(segment.word_times)
    
    # Calculate overall boundaries
    mfa_start_ms = segment_group[0].mfa_start_ms
    mfa_end_ms = segment_group[-1].mfa_end_ms
    merged_duration_ms = mfa_end_ms - mfa_start_ms
    
    # All segments should have same speaker (or pick primary)
    speaker = segment_group[0].speaker
    
    # Create merged segment
    merged = MergedSegment(
        texts=texts,
        merged_text=merged_text,
        speaker=speaker,
        original_segments=segment_group,
        mfa_start_ms=mfa_start_ms,
        mfa_end_ms=mfa_end_ms,
        merged_duration_ms=merged_duration_ms,
        word_times_merged=merged_word_times,  # ← PRESERVED ALIGNMENT
        audio_file=segment_group[0].audio_file
    )
    
    return merged


def validate_merged_alignment(merged_segment: MergedSegment) -> bool:
    """
    Validate that word-level alignment is preserved after merge.
    """
    
    # Check boundary consistency
    word_starts = [w['start_ms'] for w in merged_segment.word_times_merged]
    word_ends = [w['end_ms'] for w in merged_segment.word_times_merged]
    
    if word_starts and word_ends:
        if min(word_starts) < merged_segment.mfa_start_ms:
            return False
        
        if max(word_ends) > merged_segment.mfa_end_ms:
            return False
    
    # Check monotonic progression
    for i in range(len(merged_segment.word_times_merged) - 1):
        w1_end = merged_segment.word_times_merged[i]['end_ms']
        w2_start = merged_segment.word_times_merged[i+1]['start_ms']
        
        if w1_end > w2_start:
            return False
    
    return True


def merge_segments_to_target_duration(
    audio_segments: List,  # From ZONE 4: Audio Segmentation
    target_duration_sec: Tuple[float, float] = (10, 30),
    strategy: str = "greedy"
) -> List[MergedSegment]:
    """
    Merge consecutive audio segments to reach target duration.
    ⚠️  CRITICAL: Preserve exact word-level alignment after merge.
    
    Args:
        audio_segments: Segments from ZONE 4 (AudioSegment[])
        target_duration_sec: (min_sec, max_sec) for merged segments
        strategy: "greedy" = merge until reaching target
    
    Returns:
        List[MergedSegment] with preserved word-level alignment
    """
    
    min_duration_ms = target_duration_sec[0] * 1000
    max_duration_ms = target_duration_sec[1] * 1000
    
    merged_segments = []
    
    if strategy == "greedy":
        # Greedy merging: combine consecutive segments
        i = 0
        while i < len(audio_segments):
            current_group = [audio_segments[i]]
            current_duration = audio_segments[i].mfa_duration_ms
            
            # Add segments while under max duration
            j = i + 1
            while j < len(audio_segments) and current_duration < max_duration_ms:
                
                candidate = audio_segments[j]
                new_duration = current_duration + candidate.mfa_duration_ms
                
                if new_duration <= max_duration_ms:
                    current_group.append(candidate)
                    current_duration = new_duration
                    j += 1
                else:
                    break
            
            # Check if merged segment reaches minimum duration
            if current_duration >= min_duration_ms:
                merged = _merge_segment_group(current_group)
                merged_segments.append(merged)
                i = j
            else:
                # Segment too short, skip or log
                print(f"⚠️  Skipping too-short segment: {current_duration:.0f}ms < {min_duration_ms:.0f}ms")
                i += 1
    
    return merged_segments

In [ ]:


print("\n" + "="*80)
print("ZONE 6: MERGE SEGMENTS TO TARGET DURATION (10–30 sec)")
print("="*80)

# Merge audio segments to 10-30 second range
merged_segments = merge_segments_to_target_duration(
    audio_segments=audio_segments,  # From ZONE 4
    target_duration_sec=(10, 30),
    strategy="greedy"
)

print(f"\n✅ Created {len(merged_segments)} merged segments")

# Statistics
if merged_segments:
    durations = [m.duration_sec for m in merged_segments]
    word_counts = [m.num_words for m in merged_segments]
    
    print(f"\n📊 Merged Segments Statistics:")
    print(f"   • Count: {len(merged_segments)}")
    print(f"   • Avg duration: {np.mean(durations):.2f}s")
    print(f"   • Min duration: {np.min(durations):.2f}s")
    print(f"   • Max duration: {np.max(durations):.2f}s")
    print(f"   • Avg word count: {np.mean(word_counts):.1f}")
    
    # Validate alignment preservation
    print(f"\n🔍 Validating alignment preservation...")
    valid_count = 0
    for merged in merged_segments:
        if validate_merged_alignment(merged):
            valid_count += 1
    
    print(f"   ✅ {valid_count}/{len(merged_segments)} segments have valid alignment")
    
    # Check all segments are in target range
    in_range = sum(1 for m in merged_segments if 10 <= m.duration_sec <= 30)
    print(f"   ✅ {in_range}/{len(merged_segments)} segments in 10-30 sec range")

print("\n" + "="*80)
print("✅ ZONE 6 COMPLETE")
print("="*80)

## ZONE 7 : DATA QUALITY ANALYSIS

In [16]:
# ════════════════════════════════════════════════════════════════════════════
# ZONE 6: DATA QUALITY ANALYSIS (AVEC VAD MODEL LÉGER)
# ════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from pathlib import Path


# ════════════════════════════════════════════════════════════════════════════
# LIGHTWEIGHT VAD MODEL (Silence Detection)
# ════════════════════════════════════════════════════════════════════════════

class LightweightVAD:
    """
    Lightweight Voice Activity Detection model.
    Uses energy-based detection without neural networks.
    
    Fast, lightweight, no model dependencies needed.
    """
    
    def __init__(self, 
                 sr: int = 16000,
                 frame_length: int = 512,
                 hop_length: int = 512,
                 energy_threshold_percentile: int = 30):
        """
        Initialize VAD model.
        
        Args:
            sr: Sample rate
            frame_length: FFT window size
            hop_length: Hop length for STFT
            energy_threshold_percentile: Percentile for threshold (0-100)
        """
        self.sr = sr
        self.frame_length = frame_length
        self.hop_length = hop_length
        self.energy_threshold_percentile = energy_threshold_percentile
    
    def detect(self, y: np.ndarray) -> tuple:
        """
        Detect speech activity in audio signal.
        
        Args:
            y: Audio waveform
            
        Returns:
            (speech_ratio, speech_frames, threshold)
            - speech_ratio: Proportion of frames with speech (0-1)
            - speech_frames: Boolean array indicating speech frames
            - threshold: Energy threshold used
        """
        
        # Compute RMS energy
        rms = librosa.feature.rms(y=y, frame_length=self.frame_length, 
                                  hop_length=self.hop_length)[0]
        
        # Compute dynamic threshold based on percentile
        threshold = np.percentile(rms, self.energy_threshold_percentile)
        
        # Detect speech frames (above threshold)
        speech_frames = rms > threshold
        
        # Compute speech activity ratio
        speech_ratio = np.sum(speech_frames) / len(speech_frames) if len(speech_frames) > 0 else 0
        
        return speech_ratio, speech_frames, threshold


class EnhancedVAD(LightweightVAD):
    """
    Enhanced VAD with additional acoustic features.
    
    Combines:
    - Energy-based detection
    - Zero-crossing rate
    - Spectral flatness
    """
    
    def __init__(self, sr: int = 16000, frame_length: int = 512, hop_length: int = 512):
        super().__init__(sr, frame_length, hop_length)
    
    def detect_enhanced(self, y: np.ndarray) -> dict:
        """
        Enhanced speech detection using multiple features.
        
        Args:
            y: Audio waveform
            
        Returns:
            Dictionary with:
            - speech_ratio: Main speech activity ratio
            - energy_speech_ratio: From energy alone
            - zcr_speech_ratio: From zero-crossing rate
            - spectral_speech_ratio: From spectral flatness
            - combined_speech_ratio: Consensus decision
            - details: Dict with detailed metrics
        """
        
        # Feature 1: RMS Energy
        rms = librosa.feature.rms(y=y, frame_length=self.frame_length,
                                  hop_length=self.hop_length)[0]
        energy_threshold = np.percentile(rms, 30)
        energy_speech = rms > energy_threshold
        energy_ratio = np.sum(energy_speech) / len(energy_speech)
        
        # Feature 2: Zero-Crossing Rate (high ZCR = unvoiced/noise)
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=self.hop_length)[0]
        zcr_threshold = np.percentile(zcr, 60)  # High ZCR threshold
        zcr_speech = zcr < zcr_threshold
        zcr_ratio = np.sum(zcr_speech) / len(zcr_speech)
        
        # Feature 3: Spectral Flatness (low flatness = tonal speech)
        S = librosa.stft(y)
        freqs = librosa.fft_frequencies(sr=self.sr)
        
        # Simple spectral flatness: ratio of geometric to arithmetic mean
        magnitude = np.abs(S)
        spectral_flatness = np.zeros(magnitude.shape[1])
        
        for t in range(magnitude.shape[1]):
            spectrum = magnitude[:, t]
            if np.sum(spectrum) > 0:
                geom_mean = np.exp(np.mean(np.log(spectrum + 1e-10)))
                arith_mean = np.mean(spectrum)
                spectral_flatness[t] = geom_mean / (arith_mean + 1e-10)
        
        spectral_threshold = np.percentile(spectral_flatness, 40)
        spectral_speech = spectral_flatness < spectral_threshold
        spectral_ratio = np.sum(spectral_speech) / len(spectral_flatness)
        
        # Consensus: frames that agree on at least 2 features
        consensus = (energy_speech.astype(int) + 
                    zcr_speech.astype(int) + 
                    spectral_speech.astype(int)) >= 2
        combined_ratio = np.sum(consensus) / len(consensus)
        
        return {
            'speech_ratio': combined_ratio,  # Main metric
            'energy_speech_ratio': energy_ratio,
            'zcr_speech_ratio': zcr_ratio,
            'spectral_speech_ratio': spectral_ratio,
            'combined_speech_ratio': combined_ratio,
            'details': {
                'energy_threshold': energy_threshold,
                'zcr_threshold': zcr_threshold,
                'spectral_threshold': spectral_threshold,
                'energy_speech_frames': np.sum(energy_speech),
                'zcr_speech_frames': np.sum(zcr_speech),
                'spectral_speech_frames': np.sum(spectral_speech),
                'consensus_frames': np.sum(consensus),
                'total_frames': len(energy_speech)
            }
        }


# ════════════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION WITH VAD
# ════════════════════════════════════════════════════════════════════════════

def extract_audio_segment_features_with_vad(audio_segments, use_enhanced_vad=True):
    """
    Extract features from audio segments using VAD model for accurate speech ratio.
    
    Args:
        audio_segments: List of segment dicts from Zone 4
        use_enhanced_vad: Use enhanced VAD (True) or basic VAD (False)
        
    Returns:
        DataFrame with extracted features including VAD-based speech_activity_ratio
    """
    
    rows = []
    
    print(f"\n{'='*80}")
    print("ZONE 6: EXTRACTING FEATURES WITH VAD")
    print(f"{'='*80}\n")
    
    # Initialize VAD model
    if use_enhanced_vad:
        print("Using Enhanced VAD (multi-feature)\n")
        vad = EnhancedVAD(sr=16000)
    else:
        print("Using Lightweight VAD (energy-based)\n")
        vad = LightweightVAD(sr=16000)
    
    for i, seg in enumerate(tqdm(audio_segments, desc="Processing segments")):
        audio_path = Path(seg["output_file"]) if "output_file" in seg else Path(seg.get("audio_path", ""))
        
        features = {
            'segment_id': seg.get("original_segment_id", f"seg_{i}"),
            'speaker': seg.get("speaker", "UNK"),
            'duration_ms': seg.get("mfa_duration_ms", 0),
            'text': seg.get("text", ""),
            'n_words': len(seg.get("text", "").split()),
            'text_length': len(seg.get("text", "")),
        }
        
        # Speech rate
        if seg.get("mfa_duration_ms", 0) > 0:
            features['speech_rate_wps'] = len(seg.get("text", "").split()) / (seg.get("mfa_duration_ms", 1) / 1000)
        else:
            features['speech_rate_wps'] = 0
        
        # Load and analyze audio with VAD
        if audio_path.exists():
            try:
                y, sr = librosa.load(str(audio_path), sr=16000)
                
                # ✅ VAD-based speech activity (IMPROVED)
                if use_enhanced_vad:
                    vad_result = vad.detect_enhanced(y)
                    features['speech_activity_ratio'] = vad_result['speech_ratio']
                    features['vad_energy_ratio'] = vad_result['energy_speech_ratio']
                    features['vad_zcr_ratio'] = vad_result['zcr_speech_ratio']
                    features['vad_spectral_ratio'] = vad_result['spectral_speech_ratio']
                else:
                    speech_ratio, _, _ = vad.detect(y)
                    features['speech_activity_ratio'] = speech_ratio
                
                # RMS Energy
                rms = librosa.feature.rms(y=y)[0]
                features['energy_mean'] = np.mean(rms)
                features['energy_std'] = np.std(rms)
                
                # Zero-crossing rate
                zcr = librosa.feature.zero_crossing_rate(y)[0]
                features['zcr_mean'] = np.mean(zcr)
                
                # Dynamic range
                db = librosa.power_to_db(np.abs(librosa.stft(y))**2, ref=np.max)
                features['dynamic_range'] = np.max(db) - np.min(db)
                
                # Spectral centroid
                spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
                features['spectral_centroid'] = np.mean(spectral_centroids)
                
            except Exception as e:
                print(f"  ⚠️  Error processing {audio_path}: {e}")
                features.update({
                    'speech_activity_ratio': np.nan,
                    'energy_mean': np.nan,
                    'energy_std': np.nan,
                    'zcr_mean': np.nan,
                    'dynamic_range': np.nan,
                    'spectral_centroid': np.nan,
                })
        else:
            features.update({
                'speech_activity_ratio': np.nan,
                'energy_mean': np.nan,
                'energy_std': np.nan,
                'zcr_mean': np.nan,
                'dynamic_range': np.nan,
                'spectral_centroid': np.nan,
            })
        
        rows.append(features)
    
    df = pd.DataFrame(rows)
    print(f"\n✅ Extracted features for {len(df)} segments\n")
    return df


# ════════════════════════════════════════════════════════════════════════════
# QUALITY SCORING
# ════════════════════════════════════════════════════════════════════════════

class AudioQualityScorer:
    """
    Score audio segment quality based on acoustic and metadata features.
    Uses VAD-based speech activity detection.
    """
    
    def __init__(self):
        self.rules = {
            'speech_rate': {
                'min': 0.5,
                'max': 6,
                'penalty': 0.3,
                'reason': 'Speech rate outside normal range (0.5-6 words/sec)'
            },
            'duration': {
                'min_ms': 5000,
                'max_ms': 60000,
                'penalty': 0.2,
                'reason': 'Segment duration outside range (5-60 sec)'
            },
            'text_length': {
                'min_chars': 10,
                'penalty': 0.1,
                'reason': 'Text too short (< 10 characters)'
            },
            'speech_activity': {
                'min_ratio': 0.4,  # VAD should detect at least 40% speech
                'penalty': 0.25,
                'reason': 'Too much silence/noise (<40% speech activity by VAD)'
            },
            'energy': {
                'min_mean': 0.01,
                'penalty': 0.1,
                'reason': 'Audio too quiet'
            },
            'dynamic_range': {
                'min_db': 5.0,
                'penalty': 0.15,
                'reason': 'Poor dynamic range'
            }
        }
    
    def score(self, df):
        """Compute quality scores for each segment."""
        scores = np.ones(len(df))
        issues = {i: [] for i in range(len(df))}
        
        # Rule 1: Speech rate
        bad = (df['speech_rate_wps'] < self.rules['speech_rate']['min']) | \
              (df['speech_rate_wps'] > self.rules['speech_rate']['max'])
        scores[bad] -= self.rules['speech_rate']['penalty']
        for idx in df[bad].index:
            issues[idx].append(self.rules['speech_rate']['reason'])
        
        # Rule 2: Duration
        bad = (df['duration_ms'] < self.rules['duration']['min_ms']) | \
              (df['duration_ms'] > self.rules['duration']['max_ms'])
        scores[bad] -= self.rules['duration']['penalty']
        for idx in df[bad].index:
            issues[idx].append(self.rules['duration']['reason'])
        
        # Rule 3: Text length
        bad = df['text_length'] < self.rules['text_length']['min_chars']
        scores[bad] -= self.rules['text_length']['penalty']
        for idx in df[bad].index:
            issues[idx].append(self.rules['text_length']['reason'])
        
        # Rule 4: Speech activity (VAD-based) - IMPROVED
        if 'speech_activity_ratio' in df.columns:
            bad = (df['speech_activity_ratio'] < self.rules['speech_activity']['min_ratio']) & \
                  (df['speech_activity_ratio'].notna())
            scores[bad] -= self.rules['speech_activity']['penalty']
            for idx in df[bad].index:
                issues[idx].append(self.rules['speech_activity']['reason'])
        
        # Rule 5: Energy
        if 'energy_mean' in df.columns:
            bad = (df['energy_mean'] < self.rules['energy']['min_mean']) & \
                  (df['energy_mean'].notna())
            scores[bad] -= self.rules['energy']['penalty']
            for idx in df[bad].index:
                issues[idx].append(self.rules['energy']['reason'])
        
        # Rule 6: Dynamic range
        if 'dynamic_range' in df.columns:
            bad = (df['dynamic_range'] < self.rules['dynamic_range']['min_db']) & \
                  (df['dynamic_range'].notna())
            scores[bad] -= self.rules['dynamic_range']['penalty']
            for idx in df[bad].index:
                issues[idx].append(self.rules['dynamic_range']['reason'])
        
        return np.clip(scores, 0, 1), issues


def compute_quality_scores(df):
    """Compute quality scores using AudioQualityScorer."""
    scorer = AudioQualityScorer()
    scores, issues = scorer.score(df)
    df['quality_score'] = scores
    df['quality_issues'] = [issues[i] for i in range(len(df))]
    return df


# ════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ════════════════════════════════════════════════════════════════════════════

def plot_quality_analysis(df):
    """Plot comprehensive quality analysis."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 1. Quality score distribution
    axes[0, 0].hist(df['quality_score'], bins=50, color='#2E86AB', alpha=0.7, edgecolor='black')
    axes[0, 0].axvline(df['quality_score'].mean(), color='red', linestyle='--', linewidth=2,
                       label=f'Mean: {df["quality_score"].mean():.3f}')
    axes[0, 0].set_xlabel('Quality Score')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Quality Score Distribution')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Speech rate vs quality
    axes[0, 1].scatter(df['speech_rate_wps'], df['quality_score'],
                      c=df['quality_score'], cmap='RdYlGn', alpha=0.6, s=30)
    axes[0, 1].axhline(0.7, color='red', linestyle='--', alpha=0.5, label='Threshold')
    axes[0, 1].set_xlabel('Speech Rate (words/sec)')
    axes[0, 1].set_ylabel('Quality Score')
    axes[0, 1].set_title('Speech Rate vs Quality')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Duration distribution
    axes[0, 2].hist(df['duration_ms']/1000, bins=40, color='#A23B72', alpha=0.7, edgecolor='black')
    axes[0, 2].set_xlabel('Duration (seconds)')
    axes[0, 2].set_ylabel('Count')
    axes[0, 2].set_title('Segment Duration Distribution')
    axes[0, 2].grid(True, alpha=0.3, axis='y')
    
    # 4. Word count distribution
    axes[1, 0].hist(df['n_words'], bins=40, color='#F18F01', alpha=0.7, edgecolor='black')
    axes[1, 0].set_xlabel('Number of Words')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Words per Segment')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # 5. Speech activity (VAD) - NOW WITH VAD MODEL
    if 'speech_activity_ratio' in df.columns:
        axes[1, 1].hist(df['speech_activity_ratio'].dropna(), bins=40, color='#06A77D', 
                       alpha=0.7, edgecolor='black')
        axes[1, 1].axvline(0.4, color='red', linestyle='--', linewidth=2, label='Threshold (40%)')
        axes[1, 1].set_xlabel('Speech Activity Ratio (VAD)')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].set_title('Speech Activity (VAD-based)')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    # 6. Quality by speaker
    if 'speaker' in df.columns:
        speaker_quality = df.groupby('speaker')['quality_score'].mean().sort_values(ascending=False).head(10)
        axes[1, 2].barh(range(len(speaker_quality)), speaker_quality.values, 
                       color='#2E86AB', alpha=0.7, edgecolor='black')
        axes[1, 2].set_yticks(range(len(speaker_quality)))
        axes[1, 2].set_yticklabels(speaker_quality.index, fontsize=9)
        axes[1, 2].set_xlabel('Average Quality Score')
        axes[1, 2].set_title('Quality by Speaker (Top 10)')
        axes[1, 2].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    return fig


# ════════════════════════════════════════════════════════════════════════════
# QUALITY REPORT AND FILTERING
# ════════════════════════════════════════════════════════════════════════════

def filter_and_report_quality(df, threshold=0.7):
    """Filter segments by quality and print detailed report."""
    
    print(f"\n{'='*80}")
    print("ZONE 7: DATA QUALITY ANALYSIS & FILTERING")
    print(f"{'='*80}\n")
    
    print(f"📊 OVERALL STATISTICS")
    print(f"   Total segments analyzed: {len(df):,}")
    print(f"   Quality score range: [{df['quality_score'].min():.3f}, {df['quality_score'].max():.3f}]")
    print(f"   Mean quality score: {df['quality_score'].mean():.3f}")
    print(f"   Median quality score: {df['quality_score'].median():.3f}\n")
    
    # Speech activity statistics (VAD)
    if 'speech_activity_ratio' in df.columns:
        print(f"🎙️  SPEECH ACTIVITY (VAD-based)")
        print(f"   Mean speech ratio: {df['speech_activity_ratio'].mean():.2%}")
        print(f"   Segments with <40% speech: {(df['speech_activity_ratio'] < 0.4).sum()} ({(df['speech_activity_ratio'] < 0.4).mean()*100:.1f}%)\n")
    
    # Filter
    high_quality = df[df['quality_score'] >= threshold].copy()
    removed = df[df['quality_score'] < threshold].copy()
    
    print(f"🎯 FILTERING RESULTS (threshold: {threshold})")
    print(f"   ✅ KEPT:    {len(high_quality):6d} ({len(high_quality)/len(df)*100:5.1f}%)  HIGH-QUALITY")
    print(f"   ❌ REMOVED: {len(removed):6d} ({len(removed)/len(df)*100:5.1f}%)  LOW-QUALITY\n")
    
    # Issue analysis
    if len(removed) > 0:
        print(f"📋 WHY SEGMENTS WERE REMOVED")
        issue_counts = {}
        for issues_list in removed['quality_issues']:
            for issue in issues_list:
                issue_counts[issue] = issue_counts.get(issue, 0) + 1
        
        for issue, count in sorted(issue_counts.items(), key=lambda x: -x[1]):
            pct = count / len(removed) * 100 if len(removed) > 0 else 0
            print(f"   • {issue:60s}: {count:5d} ({pct:5.1f}%)")
    
    # Stats on high-quality segments
    if len(high_quality) > 0:
        print(f"\n✅ HIGH-QUALITY SEGMENTS STATISTICS")
        print(f"   Avg duration: {high_quality['duration_ms'].mean()/1000:.2f}s")
        print(f"   Avg words: {high_quality['n_words'].mean():.1f}")
        print(f"   Avg speech rate: {high_quality['speech_rate_wps'].mean():.2f} words/sec")
        if 'speech_activity_ratio' in high_quality.columns:
            print(f"   Avg speech activity (VAD): {high_quality['speech_activity_ratio'].mean():.2%}")
    
    print(f"\n{'='*80}\n")
    
    return high_quality, removed


# ════════════════════════════════════════════════════════════════════════════
# MAIN ZONE 6 EXECUTION
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("ZONE 7: DATA QUALITY ANALYSIS (WITH VAD)")
print("="*80)

# Extract features with VAD
features_df = extract_audio_segment_features_with_vad(
    audio_segments=merged_segments,  # From ZONE 7
    use_enhanced_vad=True  # Use enhanced VAD with multi-feature detection
)

# Compute quality scores
features_df = compute_quality_scores(features_df)

# Filter and report
quality_segments, removed_segments = filter_and_report_quality(
    features_df,
    threshold=0.7  # Keep segments with quality score >= 0.7
)

# Visualize
fig = plot_quality_analysis(features_df)
plt.show()

print(f" ZONE 7 COMPLETE\n")
print(f"   Input segments (from Zone 6):  {len(merged_segments)}")
print(f"   Output segments (quality filtered): {len(quality_segments)}")

## Zone 8 : Whisper Baseline Evaluation Compute of the WER before fine-tuning

In [21]:
@dataclass
class TranscriptionResult:
    segment_id: str
    speaker: str
    file_name: str
    audio_path: str
    ground_truth: str
    whisper_prediction: str
    duration_ms: int
    wer: float
    confidence: float = 0.0


class WhisperEvaluator:
    """Evaluate Whisper baseline on children voices"""

    def __init__(self, model_name: str = "base"):
        print(f"Loading Whisper '{model_name}'...")
        self.model = whisper.load_model(model_name)
        print("Loaded\n")

    def transcribe(self, audio_path: Path) -> Dict:
        if not audio_path.exists():
            return {"text": "", "confidence": 0.0}
        try:
            result = self.model.transcribe(str(audio_path), language="fr", verbose=False)
            return {"text": result["text"].strip(), "confidence": result.get("confidence", 0.0)}
        except:
            return {"text": "", "confidence": 0.0}

    @staticmethod
    def calculate_wer(ground_truth: str, prediction: str) -> float:
        if not ground_truth.strip():
            return 0.0 if not prediction.strip() else 1.0
        return compute_wer(ground_truth, prediction)

    def evaluate_children(self, audio_segments: List[Dict], speakers_info: Dict,
                         sample_size: int = None) -> List[TranscriptionResult]:
        """Evaluate only children speakers"""

        print("="*70)
        print("STEP 4: WHISPER BASELINE EVALUATION (CHILDREN ONLY)")
        print("="*70)

        # Filter children only
        children_segments = [s for s in audio_segments
                            if speakers_info.get(s["speaker"]) in CHILD_ROLES]

        if sample_size:
            children_segments = children_segments[:sample_size]

        print(f"\nEvaluating {len(children_segments)} children segments\n")

        results = []
        for i, seg in enumerate(children_segments):
            audio_path = Path(seg["audio_path"])
            transcription = self.transcribe(audio_path)
            wer = self.calculate_wer(seg["text"], transcription["text"])

            results.append(TranscriptionResult(
                segment_id=seg["segment_id"],
                speaker=seg["speaker"],
                file_name=seg["file_name"],
                audio_path=seg["audio_path"],
                ground_truth=seg["text"],
                whisper_prediction=transcription["text"],
                duration_ms=seg["duration_ms"],
                wer=wer,
                confidence=transcription["confidence"]
            ))

            if (i + 1) % 50 == 0:
                avg_wer = sum(r.wer for r in results) / len(results)
                print(f"  ✓ {i + 1}/{len(children_segments)} | Avg WER: {avg_wer:.3f}")

        return results


## Zone 9: WER Report

In [22]:
def print_wer_report(results: List[TranscriptionResult], speakers_info: Dict):
    """Print WER statistics"""

    wers = [r.wer for r in results]

    print("\n" + "="*70)
    print("STEP 5: WER STATISTICS (CHILDREN ONLY)")
    print("="*70)

    print(f"\nGlobal:")
    print(f"   Total segments:  {len(results)}")
    print(f"   Avg WER:         {sum(wers) / len(wers):.3f}")
    print(f"   Min WER:         {min(wers):.3f}")
    print(f"   Max WER:         {max(wers):.3f}")
    print(f"   Median WER:      {sorted(wers)[len(wers)//2]:.3f}")

    print(f"\n   Distribution:")
    for low, high in [(0.0, 0.1), (0.1, 0.3), (0.3, 0.5), (0.5, 1.0)]:
        count = sum(1 for w in wers if low <= w < high)
        pct = (count / len(wers)) * 100
        print(f"      {low:.1f}-{high:.1f}: {count:4d} ({pct:5.1f}%)")

    # By speaker
    by_speaker = {}
    for r in results:
        by_speaker.setdefault(r.speaker, []).append(r.wer)

    print(f"\n👥 By speaker ({len(by_speaker)}):")
    for speaker in sorted(by_speaker.keys()):
        wers_sp = by_speaker[speaker]
        avg = sum(wers_sp) / len(wers_sp)
        print(f"      {speaker:15} {len(wers_sp):4d} segments | WER: {avg:.3f}")

    print("\n" + "="*70 + "\n")


In [23]:
print("\n" + "="*70)
print("ZONE 6: WHISPER EVALUATION (HIGH-QUALITY SEGMENTS ONLY)")
print("="*70)

    # Filter audio_segments to only include high-quality ones
high_quality_ids = set(high_quality_df['speaker'].unique())
filtered_audio_segments = [
    seg for seg in audio_segments
    if seg['speaker'] in high_quality_ids  # Or use segment_id if available
]

print(f"\n   📊 Using {len(filtered_audio_segments)}/{len(audio_segments)} segments for Whisper")
print(f"      (Filtered: {len(audio_segments) - len(filtered_audio_segments)} low-quality segments removed)")

evaluator = WhisperEvaluator(CONFIG["whisper_model"])
results = evaluator.evaluate_children(
    filtered_audio_segments,  # HIGH-QUALITY ONLY!
    speakers_info,
    CONFIG["sample_size"]
)

print_wer_report(results, speakers_info)



ZONE 6: WHISPER EVALUATION (HIGH-QUALITY SEGMENTS ONLY)

   📊 Using 79/79 segments for Whisper
      (Filtered: 0 low-quality segments removed)
Loading Whisper 'base'...
Loaded

STEP 4: WHISPER BASELINE EVALUATION (CHILDREN ONLY)

Evaluating 25 children segments


100%|██████████| 1321/1321 [00:08<00:00, 149.89frames/s]
0frames [00:00, ?frames/s]
0frames [00:00, ?frames/s]
100%|██████████| 1588/1588 [00:05<00:00, 316.50frames/s]
0frames [00:00, ?frames/s]
0frames [00:00, ?frames/s]
100%|██████████| 554/554 [00:01<00:00, 442.52frames/s]


STEP 5: WER STATISTICS (CHILDREN ONLY)

Global:
   Total segments:  25
   Avg WER:         2.237
   Min WER:         0.917
   Max WER:         7.333
   Median WER:      1.500

   Distribution:
      0.0-0.1:    0 (  0.0%)
      0.1-0.3:    0 (  0.0%)
      0.3-0.5:    0 (  0.0%)
      0.5-1.0:    1 (  4.0%)

👥 By speaker (10):
      CAR                1 segments | WER: 1.000
      CHI                1 segments | WER: 3.000
      KLO                3 segments | WER: 1.515
      LUS                8 segments | WER: 2.496
      MAI                4 segments | WER: 3.458
      MAT                3 segments | WER: 1.756
      NIN                2 segments | WER: 1.256
      RIT                1 segments | WER: 3.800
      SAR                1 segments | WER: 1.000
      WIL                1 segments | WER: 1.000



## Zone 10: Create Training Dataset

In [24]:
# When i am done i have to analyze the datas before creation of the training set


class DatasetBuilder:
    """Create train/test splits for fine-tuning"""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def create_dataset(self, results: List[TranscriptionResult], train_ratio: float = 0.8):
        """Create JSONL + metadata files"""

        print("="*70)
        print("STEP 6: CREATE TRAINING DATASET")
        print("="*70)

        split_idx = int(len(results) * train_ratio)
        train = results[:split_idx]
        test = results[split_idx:]

        print(f"\nDataset split:")
        print(f"   Total:   {len(results)} segments")
        print(f"   Train:   {len(train)} segments ({train_ratio*100:.0f}%)")
        print(f"   Test:    {len(test)} segments ({(1-train_ratio)*100:.0f}%)")

        # Save JSONL (for fine-tuning)
        self._save_jsonl(train, self.output_dir / "train.jsonl")
        self._save_jsonl(test, self.output_dir / "eval.jsonl")

        # Save metadata JSON (for analysis)
        self._save_metadata(train, self.output_dir / "train_metadata.json")
        self._save_metadata(test, self.output_dir / "eval_metadata.json")

        print(f"\nDataset created in {self.output_dir}\n")

    def _save_jsonl(self, results: List[TranscriptionResult], output_file: Path):
        """Save as JSONL for Whisper"""
        with open(output_file, "w", encoding="utf-8") as f:
            for r in results:
                entry = {"audio": r.audio_path, "text": r.ground_truth, "language": "fr"}
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        print(f"   ✓ {output_file.name} ({len(results)} segments)")

    def _save_metadata(self, results: List[TranscriptionResult], output_file: Path):
        """Save complete metadata"""
        data = [asdict(r) for r in results]
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        print(f"   ✓ {output_file.name}")

In [25]:

    # ════════════════════════════════════════════════════════════════════════
    # ZONE 7: Create Training Dataset
    # ════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("ZONE 7: CREATE TRAINING DATASET")
print("="*70)

builder = DatasetBuilder(CONFIG["output_dir"] / "training_dataset")
builder.create_dataset(results, CONFIG["train_ratio"])



ZONE 7: CREATE TRAINING DATASET
STEP 6: CREATE TRAINING DATASET

Dataset split:
   Total:   25 segments
   Train:   20 segments (80%)
   Test:    5 segments (20%)
   ✓ train.jsonl (20 segments)
   ✓ eval.jsonl (5 segments)
   ✓ train_metadata.json
   ✓ eval_metadata.json

Dataset created in /content/drive/MyDrive/asr/output/whisper_children_dataset/training_dataset
